<a href="https://colab.research.google.com/github/komazawa-deep-learning/komazawa-deep-learning.github.io/blob/master/2026notebooks/2026ai_lect04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 人工知能 I 第04回：ロジスティック回帰と機械学習の背骨

**担当**: 浅川伸一 | **2026年度 前期**

---

## 本日の構成

| セクション | 内容 | 重点 |
|-----------|------|------|
| 1 | 線形回帰が分類に使えない理由 | 概念 |
| 2 | Sigmoid 関数 | 概念 + 実装 |
| 3 | 確率・尤度・最尤推定 | **概念中心** |
| 4 | 交差エントロピー損失 | 概念 + 実装 |
| 5 | 最小二乗と最尤の関係 | **概念中心** |
| 6 | 勾配降下法：学習の直観 | 概念 + 実装 |
| 7 | 過学習と L2 正則化 | 実装 |
| 8 | 実データ（乳がん）での総合演習 | 実装 |

**本日の核心**：機械学習が「どのようにデータから学ぶか」を理解する

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, roc_auc_score, ConfusionMatrixDisplay)
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from scipy.special import comb

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

try:
    import japanize_matplotlib
except ImportError:
    !pip install japanize_matplotlib
    import japanize_matplotlib


def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def cross_entropy(y_true, y_pred_proba, eps=1e-9):
    p = np.clip(y_pred_proba, eps, 1 - eps)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

print('環境準備完了')

環境準備完了


---
## 1. なぜ線形回帰で分類できないか

線形回帰を分類に使うと何が起きるかを確認します。

In [ ]:
np.random.seed(0)
n = 40
x_neg = np.random.normal(1, 0.8, n)
x_pos = np.random.normal(3, 0.8, n)
X_demo = np.concatenate([x_neg, x_pos]).reshape(-1, 1)
y_demo = np.array([0]*n + [1]*n, dtype=float)

lr_cls = LinearRegression().fit(X_demo, y_demo)
x_plot = np.linspace(-1, 6, 200).reshape(-1, 1)
y_lr   = lr_cls.predict(x_plot)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(X_demo, y_demo, c=y_demo, cmap='bwr', s=50,
                alpha=0.6, edgecolors='k', linewidth=0.5)
axes[0].plot(x_plot, y_lr, 'k-', linewidth=2, label='線形回帰')
axes[0].axhline(0.5, color='green', linewidth=1.5, linestyle='-.', label='閾値 0.5')
axes[0].set_xlabel('勉強時間'); axes[0].set_ylabel('合格(1)/不合格(0)')
axes[0].set_title('線形回帰で分類を試みる\n→ 出力が [0,1] を超える', fontsize=12)
axes[0].legend()

# 外れ値を追加すると決定境界が崩壊
x_outlier = np.array([[10.0],[11.0],[12.0]])
y_outlier  = np.array([1., 1., 1.])
X_ext = np.vstack([X_demo, x_outlier])
y_ext = np.concatenate([y_demo, y_outlier])
lr_ext = LinearRegression().fit(X_ext, y_ext)
x_plot2 = np.linspace(-1, 13, 300).reshape(-1, 1)

axes[1].scatter(X_ext, y_ext, c=y_ext, cmap='bwr', s=50,
                alpha=0.6, edgecolors='k', linewidth=0.5)
axes[1].plot(x_plot2, lr_cls.predict(x_plot2), 'b--', linewidth=2, label='外れ値なし', alpha=0.7)
axes[1].plot(x_plot2, lr_ext.predict(x_plot2), 'r-',  linewidth=2, label='外れ値あり')
axes[1].axhline(0.5, color='green', linewidth=1.5, linestyle='-.')
axes[1].set_xlabel('勉強時間')
axes[1].set_title('外れ値を追加すると決定境界が崩壊\n→ 線形回帰は分類に不適', fontsize=12)
axes[1].legend()

plt.tight_layout(); plt.show()
print(f'線形回帰の出力範囲: [{y_lr.min():.3f}, {y_lr.max():.3f}]')
print('→ 確率として解釈できない')

---
## 2. Sigmoid 関数

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \quad z = wx + b$$

出力を常に $(0, 1)$ に収めることで確率として解釈できる。

In [ ]:
z = np.linspace(-8, 8, 300)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(z, sigmoid(z), 'steelblue', linewidth=3)
axes[0].axhline(0.5, color='red', linestyle='--', linewidth=1.2, label='閾値 = 0.5')
for z_val, label in [(-4,'z=-4\n(p≈0.02)'), (0,'z=0\n(p=0.5)'), (4,'z=4\n(p≈0.98)')]:
    axes[0].scatter(z_val, sigmoid(z_val), s=100, zorder=5, color='tomato')
    axes[0].annotate(label, (z_val, sigmoid(z_val)),
                     textcoords='offset points', xytext=(5,10), fontsize=9)
axes[0].set_xlabel('z = wx + b', fontsize=11)
axes[0].set_ylabel('σ(z) = P(y=1|x)', fontsize=11)
axes[0].set_title('Sigmoid 関数\n出力は常に (0,1) ← 確率として解釈できる', fontsize=12)
axes[0].legend()

logit = LogisticRegression(C=10).fit(X_demo, y_demo)
w_l, b_l = logit.coef_[0,0], logit.intercept_[0]
prob = sigmoid(w_l * x_plot.ravel() + b_l)

axes[1].scatter(X_demo, y_demo, c=y_demo, cmap='bwr', s=50,
                alpha=0.6, edgecolors='k', linewidth=0.5)
axes[1].plot(x_plot, prob, 'steelblue', linewidth=3, label='P(合格|時間)')
axes[1].axhline(0.5, color='red', linestyle='--', linewidth=1.2, label='決定境界')
boundary = -b_l / w_l
axes[1].axvline(boundary, color='green', linestyle='-.', linewidth=1.5,
                label=f'x* = {boundary:.2f}')
axes[1].set_xlabel('勉強時間', fontsize=11)
axes[1].set_ylabel('合格確率', fontsize=11)
axes[1].set_title('ロジスティック回帰\n→ 確率として解釈できる', fontsize=12)
axes[1].legend()
plt.tight_layout(); plt.show()
print(f'w = {w_l:.4f},  b = {b_l:.4f}')
print(f'決定境界: {boundary:.2f} 時間')

---
## 3. 確率・尤度・最尤推定（概念中心）

- **確率**：パラメータ固定 → データの起きやすさ
- **尤度**：データ固定 → パラメータのもっともらしさ
- **最尤推定（MLE）**：尤度を最大化するパラメータを選ぶ

コイン7回中5回表：$L(p) = \binom{7}{5} p^5 (1-p)^2$

In [ ]:
n_toss, n_head = 7, 5
p_vals = np.linspace(0.001, 0.999, 500)
likelihood = comb(n_toss, n_head) * p_vals**n_head * (1-p_vals)**(n_toss-n_head)
p_mle = n_head / n_toss

plt.figure(figsize=(8, 4))
plt.plot(p_vals, likelihood, 'steelblue', linewidth=2)
plt.axvline(p_mle, color='red', linestyle='--', linewidth=2,
            label=f'最尤推定値 p̂ = {p_mle:.3f}')
plt.scatter(p_mle, max(likelihood), s=120, color='red', zorder=5)
plt.xlabel('表の確率 p', fontsize=11)
plt.ylabel('尤度 L(p)', fontsize=11)
plt.title(f'コイン {n_toss} 回中 {n_head} 回表\n最尤推定：尤度を最大にする p を選ぶ', fontsize=12)
plt.legend(); plt.tight_layout(); plt.show()
print(f'最尤推定量 p̂ = {n_head}/{n_toss} = {p_mle:.4f}')
print('→ 標本平均と一致（直観と合致）')

---
## 4. 交差エントロピー損失

$$L = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\right]$$

In [ ]:
p_range = np.linspace(0.001, 0.999, 500)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 交差エントロピーの直観
axes[0].plot(p_range, -np.log(p_range),   'steelblue', linewidth=2, label='-log(p̂)  [y=1の損失]')
axes[0].plot(p_range, -np.log(1-p_range), 'tomato',    linewidth=2, label='-log(1-p̂) [y=0の損失]')
axes[0].set_ylim(0, 5)
axes[0].set_xlabel('予測確率 p̂', fontsize=11)
axes[0].set_ylabel('損失', fontsize=11)
axes[0].set_title('交差エントロピーの直観\n誤った確信に対して損失が急増する', fontsize=12)
axes[0].legend()

# 交差エントロピー vs MSE
axes[1].plot(p_range, -np.log(p_range),       'steelblue', linewidth=2, label='交差エントロピー (y=1)')
axes[1].plot(p_range, (1-p_range)**2,          'steelblue', linewidth=2, label='MSE (y=1)', linestyle='--')
axes[1].set_ylim(0, 5)
axes[1].set_xlabel('予測確率 p̂', fontsize=11)
axes[1].set_ylabel('損失', fontsize=11)
axes[1].set_title('交差エントロピー vs MSE\n→ 交差エントロピーは誤りに強いペナルティ', fontsize=12)
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# 交差エントロピーの手動計算
y_true = np.array([1, 1, 0, 0])
examples = {
    '優秀な予測':   np.array([0.9, 0.8, 0.1, 0.2]),
    '中程度の予測': np.array([0.6, 0.55, 0.4, 0.45]),
    '悪い予測':     np.array([0.1, 0.2, 0.9, 0.8]),
}
print('=' * 50)
print('正解ラベル:', y_true)
print('=' * 50)
for name, proba in examples.items():
    loss = cross_entropy(y_true, proba)
    print(f'{name}: CE = {loss:.4f}')

---
## 5. 最小二乗と最尤推定の関係（概念）

| 損失関数 | 確率分布の仮定 |
|---------|---------------|
| MSE（二乗損失） | 正規分布ノイズの MLE |
| 交差エントロピー | ベルヌーイ分布の MLE |

**損失関数の選択 = データの分布に関する仮定**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 左：正規分布 → MSE（模式図）
mu_true, sigma = 2.0, 0.6
x_r = np.linspace(-1, 5, 300)
y_gauss = np.exp(-(x_r-mu_true)**2/(2*sigma**2)) / (sigma*np.sqrt(2*np.pi))
axes[0].plot(x_r, y_gauss, 'steelblue', linewidth=2)
axes[0].axvline(mu_true, color='red', linestyle='--', linewidth=1.5, label='予測値 ŷ')
for yi in [0.5, 1.8, 3.0]:
    p_i = np.exp(-(yi-mu_true)**2/(2*sigma**2))/(sigma*np.sqrt(2*np.pi))
    axes[0].scatter(yi, 0.03, s=100, color='tomato', zorder=5)
    axes[0].plot([yi,yi],[0,p_i],'tomato',linewidth=1.5,alpha=0.7)
axes[0].set_xlabel('y', fontsize=11)
axes[0].set_ylabel('確率密度', fontsize=11)
axes[0].set_title('線形回帰 + 正規分布ノイズ\n→ 最尤推定 ≡ MSE 最小化', fontsize=12)
axes[0].legend()

# 右：ベルヌーイ分布 → 交差エントロピー
p_vals2 = np.linspace(0.001, 0.999, 400)
log_lik = np.log(p_vals2) + np.log(p_vals2) + np.log(1-p_vals2)  # y=1,1,0
neg_ll  = -log_lik / 3
p_mle2  = 2/3
axes[1].plot(p_vals2, neg_ll, 'tomato', linewidth=2)
axes[1].axvline(p_mle2, color='red', linestyle='--', linewidth=2, label=f'MLE: p̂={p_mle2:.2f}')
min_loss = -log_lik[np.argmin(neg_ll)]/3
axes[1].scatter(p_mle2, neg_ll[np.argmin(neg_ll)], s=120, color='red', zorder=5)
axes[1].set_xlabel('予測確率 p̂', fontsize=11)
axes[1].set_ylabel('−log尤度（交差エントロピー）', fontsize=11)
axes[1].set_title('ロジスティック回帰 + ベルヌーイ分布\n→ 最尤推定 ≡ 交差エントロピー最小化', fontsize=12)
axes[1].legend()
plt.tight_layout(); plt.show()
print('MSE最小化 = 正規分布ノイズを仮定した最尤推定')
print('CE最小化  = ベルヌーイ分布を仮定した最尤推定')

---
## 6. 勾配降下法：モデルの「適応」

$$w \leftarrow w - \eta \frac{\partial L}{\partial w}$$

**心理学との対比**：罰（誤差信号）に従ってパラメータを調整する過程は
試行錯誤学習やΔ則と構造的に同一。

In [ ]:
def loss_func(w): return (w-2)**2 + 0.5*np.sin(3*w)
def grad_func(w): return 2*(w-2) + 1.5*np.cos(3*w)

lrs = [0.05, 0.3, 0.9]
w_range = np.linspace(-2.5, 4.5, 500)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, lr in zip(axes, lrs):
    w_hist = [-1.5]
    w = -1.5
    for _ in range(30):
        w = w - lr * grad_func(w)
        w_hist.append(w)
    w_hist = np.array(w_hist)
    L_hist = loss_func(w_hist)
    ax.plot(w_range, loss_func(w_range), 'steelblue', linewidth=2, alpha=0.5)
    ax.scatter(w_hist[:-1], L_hist[:-1], s=30, color='tomato', zorder=5, alpha=0.7)
    ax.scatter(w_hist[0],  L_hist[0],  s=120, color='green', zorder=6, label='開始')
    ax.scatter(w_hist[-1], L_hist[-1], s=120, color='red', zorder=6, marker='*', label='終着')
    for i in range(min(8, len(w_hist)-1)):
        ax.annotate('', xy=(w_hist[i+1], L_hist[i+1]),
                    xytext=(w_hist[i], L_hist[i]),
                    arrowprops=dict(arrowstyle='->', color='tomato', lw=1.2))
    label = '適切' if lr == 0.3 else ('遅い' if lr < 0.3 else '発散の危険')
    ax.set_title(f'η = {lr} → {label}', fontsize=12)
    ax.set_xlabel('パラメータ w'); ax.set_ylabel('損失 L(w)')
    ax.legend(fontsize=8); ax.set_ylim(-1, 10)

plt.suptitle('勾配降下法：学習率の影響', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
def logistic_gd_step(X, y, w, b, lr=0.1):
    n = len(y)
    p_hat  = sigmoid(X @ w + b)
    error  = p_hat - y
    grad_w = X.T @ error / n
    grad_b = error.mean()
    return w - lr*grad_w, b - lr*grad_b, cross_entropy(y, p_hat)

w_gd = np.array([0.0])
b_gd = 0.0
loss_history = []
for _ in range(300):
    w_gd, b_gd, loss = logistic_gd_step(X_demo, y_demo, w_gd, b_gd, lr=0.5)
    loss_history.append(loss)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(loss_history, 'steelblue', linewidth=2)
axes[0].set_xlabel('ステップ数', fontsize=11)
axes[0].set_ylabel('交差エントロピー損失', fontsize=11)
axes[0].set_title('学習曲線\n損失が徐々に低下 = 「学習」の進行', fontsize=12)

logit_ref = LogisticRegression(C=10).fit(X_demo, y_demo)
w_ref, b_ref = logit_ref.coef_[0,0], logit_ref.intercept_[0]
prob_gd  = sigmoid(w_gd[0] * x_plot.ravel() + b_gd)
prob_sk  = sigmoid(w_ref  * x_plot.ravel() + b_ref)

axes[1].scatter(X_demo, y_demo, c=y_demo, cmap='bwr', s=50,
                alpha=0.6, edgecolors='k', linewidth=0.5)
axes[1].plot(x_plot, prob_gd, 'steelblue', linewidth=2.5, label='手動 GD (300step)')
axes[1].plot(x_plot, prob_sk, 'r--', linewidth=1.5, label='sklearn（参考）')
axes[1].axhline(0.5, color='gray', linestyle=':', linewidth=1)
axes[1].set_xlabel('x'); axes[1].set_ylabel('P(y=1|x)')
axes[1].set_title('手動 GD vs sklearn', fontsize=12)
axes[1].legend()
plt.tight_layout(); plt.show()
print(f'手動 GD:  w={w_gd[0]:.4f}, b={b_gd:.4f}')
print(f'sklearn: w={w_ref:.4f}, b={b_ref:.4f}')

---
## 7. 過学習と L2 正則化

$$L_{\text{reg}} = \text{交差エントロピー} + \alpha \sum_j w_j^2$$

sklearn では `C = 1/α`（Cが小さいほど正則化が強い）

In [ ]:
data   = load_breast_cancer()
print(data.DESCR)

In [ ]:
data   = load_breast_cancer()
X_bc   = StandardScaler().fit_transform(data.data)
y_bc   = data.target
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)

C_vals = np.logspace(-4, 4, 40)
train_accs, test_accs, coef_norms = [], [], []
for C in C_vals:
    m = LogisticRegression(C=C, max_iter=2000).fit(X_tr, y_tr)
    train_accs.append(m.score(X_tr, y_tr))
    test_accs.append(m.score(X_te, y_te))
    coef_norms.append(np.linalg.norm(m.coef_))

best_C = C_vals[np.argmax(test_accs)]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].semilogx(C_vals, train_accs, 'steelblue', linewidth=2, label='訓練精度')
axes[0].semilogx(C_vals, test_accs,  'tomato',    linewidth=2, label='テスト精度')
axes[0].axvline(best_C, color='gray', linestyle='--', linewidth=1.5,
                label=f'最良 C={best_C:.3f}')
axes[0].set_xlabel('C（正則化の逆数）', fontsize=11)
axes[0].set_ylabel('正解率', fontsize=11)
axes[0].set_title('C と訓練/テスト精度', fontsize=12)
axes[0].legend()

axes[1].semilogx(C_vals, coef_norms, 'green', linewidth=2)
axes[1].axvline(best_C, color='gray', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('C（正則化の逆数）', fontsize=11)
axes[1].set_ylabel('係数ノルム ||w||', fontsize=11)
axes[1].set_title('C と係数の大きさ', fontsize=12)
plt.tight_layout(); plt.show()
print(f'最良 C: {best_C:.4f}, テスト精度: {max(test_accs):.4f}')

---
## 8. 総合演習：混同行列・ROC・係数の解釈

In [ ]:
best_model = LogisticRegression(C=best_C, max_iter=2000).fit(X_tr, y_tr)
y_pred  = best_model.predict(X_te)
y_proba = best_model.predict_proba(X_te)[:, 1]

print('=== 分類レポート ===')
print(classification_report(y_te, y_pred, target_names=['悪性(0)','良性(1)']))
print(f'AUC = {roc_auc_score(y_te, y_proba):.4f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ConfusionMatrixDisplay(confusion_matrix(y_te, y_pred),
    display_labels=['悪性','良性']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('混同行列', fontsize=12)

fpr, tpr, _ = roc_curve(y_te, y_proba)
auc = roc_auc_score(y_te, y_proba)
axes[1].plot(fpr, tpr, 'steelblue', linewidth=2, label=f'AUC={auc:.4f}')
axes[1].plot([0,1],[0,1],'gray',linestyle='--',linewidth=1)
axes[1].set_xlabel('偽陽性率'); axes[1].set_ylabel('真陽性率')
axes[1].set_title('ROC 曲線', fontsize=12); axes[1].legend()

coef = best_model.coef_[0]
top_idx = np.argsort(np.abs(coef))[-10:]
colors = ['tomato' if c<0 else 'steelblue' for c in coef[top_idx]]
axes[2].barh(data.feature_names[top_idx], coef[top_idx], color=colors)
axes[2].axvline(0, color='black', linewidth=0.8)
axes[2].set_title('係数の重要度（上位10）', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
print('=' * 60)
print('  第4回まとめ：ロジスティック回帰と機械学習の背骨')
print('=' * 60)
print('''
【ロジスティック回帰】
  P(y=1|x) = σ(wx+b),  σ(z) = 1/(1+exp(-z))

【尤度・最尤推定（MLE）】
  尤度：データ固定時のパラメータのもっともらしさ
  MLE：尤度を最大化するパラメータを選ぶ

【損失関数 ↔ 確率分布】
  MSE         ← 正規分布ノイズの MLE
  交差エントロピー ← ベルヌーイ分布の MLE

【勾配降下法】
  w ← w - η ∂L/∂w
  心理学的「学習」のアナロジー：誤差に従ってパラメータを更新

【L2 正則化】
  L_reg = CE + α||w||²,  sklearn では C = 1/α
  C小 → 強い正則化 → 過学習を防ぐ
''')
print('次回：決定木・Random Forest・ブースティング')

---
## 準備学習課題

**課題1**（必須・60分）  
SVM と Random Forest の違いを，学習の仕組み・決定境界の特徴・ハイパーパラメータの観点から 1段落（100〜150字）で説明せよ。

**課題2**（任意）  
交差エントロピーとMSEのどちらがロジスティック回帰に適しているか，勾配の大きさの観点から数値例を挙げて説明せよ。

---
### 解答欄

**課題1**:


**課題2（任意）**:


---
## 参考文献

1. Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.
2. Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press.
3. scikit-learn: LogisticRegression — https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

---
*人工知能I 第4回実習ノートブック | 担当：浅川伸一*